In [12]:


import os
import random
import numpy as np
import tensorflow as tf
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

SEED = 1234

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

from project.loader import Loader
from project.CROP.models.crop2 import Crop2 
from project.CROP.models.crop3_estimation_correction import Crop3EstimationCorrection
from project.CROP.models.crop3_mask_correction import Crop3MaskCorrection

dataset = "fashion"

inter=128
lat=64
data = Loader.data(dataset=dataset)
predictor = Loader.predictor(dataset=dataset)

x_train = data["x_train"]
x_train_1 = data["x_train_1"]
y_train = data["y_train"]
y_train_1 = data["y_train_1"]
ITERATIONS = 30

c = Crop3MaskCorrection(predictor=predictor,lat=lat,inter=inter,dataset=dataset)

def objective(trial):

    gamma = trial.suggest_float("gamma", -5,5)
    delta = trial.suggest_float("delta", 0,10)
    threshold = trial.suggest_int("threshold",1,ITERATIONS)
    n_images=2000
    try:        
        metrics = c.unmix(
            x_train[:n_images],
            x_train_1[:n_images],
            y_train[:n_images],
            y_train_1[:n_images],
            params={"gamma":gamma,"delta":delta,"threshold":threshold},
            iterations=ITERATIONS,
            show_metrics=False
        )
        met = float(metrics["acc_both"])
        return  met

    except Exception as e:
        print(f"Error: {e}")
        return float("inf")
    



Usando fashion como dataset


In [13]:
import optuna


# Creamos un estudio de minimización
study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials=200) # Probamos 30 combinaciones

# Mostramos los mejores resultados
print("\n📊 Mejores hiperparámetros encontrados:")
print(study.best_trials)

#mejores parametros para Crop3MaskCorrection  optimizando acc_both.

#params={'delta': 3.817760761786454}

#mejores parametros para Crop3MaskCorrection  optimizando SSIM.

#params={'gamma': 1.4197414603476595, 'delta': 5.593244163014582, 'threshold': 11}
#----------------------------------------------------------------------

#mejores parametros para Crop3EstimationCorrection  optimizando acc_both.
#params={'gamma': 1.265852455556597, 'delta': 0.3429787390624981}

#mejores parametros para Crop3EstimationCorrection  optimizando SSIM.
#params={'gamma': 1.5420520651370637, 'delta': 0.4771720136717561}



[I 2026-05-25 16:36:48,983] A new study created in memory with name: no-name-585f0692-c776-4430-b789-09cce5f71394
[I 2026-05-25 16:36:56,762] Trial 0 finished with value: 0.3785 and parameters: {'gamma': -4.167243512905577, 'delta': 9.920791845014248, 'threshold': 4}. Best is trial 0 with value: 0.3785.
[I 2026-05-25 16:37:04,261] Trial 1 finished with value: 0.3975 and parameters: {'gamma': 3.574585702287594, 'delta': 8.94392237859075, 'threshold': 9}. Best is trial 1 with value: 0.3975.
[I 2026-05-25 16:37:10,589] Trial 2 finished with value: 0.3515 and parameters: {'gamma': -4.20370752359875, 'delta': 9.643286552378195, 'threshold': 29}. Best is trial 1 with value: 0.3975.
[I 2026-05-25 16:37:17,638] Trial 3 finished with value: 0.253 and parameters: {'gamma': 2.2303834418262856, 'delta': 6.0501702894507865, 'threshold': 11}. Best is trial 1 with value: 0.3975.
[I 2026-05-25 16:37:25,209] Trial 4 finished with value: 0.448 and parameters: {'gamma': 4.129797971257334, 'delta': 5.7796


📊 Mejores hiperparámetros encontrados:
[FrozenTrial(number=171, state=1, values=[0.677], datetime_start=datetime.datetime(2026, 5, 25, 16, 52, 44, 30767), datetime_complete=datetime.datetime(2026, 5, 25, 16, 52, 54, 22671), params={'gamma': 0.7959117086604368, 'delta': 2.63671732778962, 'threshold': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'gamma': FloatDistribution(high=5.0, log=False, low=-5.0, step=None), 'delta': FloatDistribution(high=10.0, log=False, low=0.0, step=None), 'threshold': IntDistribution(high=30, log=False, low=1, step=1)}, trial_id=171, value=None)]


In [ ]:
n_image=10
labels=["0","1","2","3","4","5","6","7","8","9","0"]


#parámetros optimizados para ssim
ssim_params={'bias': 0.3490758398711943, 'slope': 23.79981809771706, 'alpha_1': -2.5004348572265864, 'alpha_2': -16.418488717733673, 'gamma': 0.2612293606781744}
#parámetros optimizados para recon_bpsnr
recon_bpsnr_params={'bias': 0.1753106428802752, 'slope': 20.088006577214188, 'alpha_1': -4.3120902863188775, 'alpha_2': -31.469765177549906, 'gamma': 0.7367839585631266}
#parámetros optimizados para mask_bpsnr
mask_bpsnr_params={'bias': 0.4032196468954787, 'slope': 10.773311686035228, 'alpha_1': -4.487552303478556, 'alpha_2': -7.255734630588293, 'gamma': 0.44933267965195817}#parámetros optimizados para acc_both
#parámetros optimizados para acc_both
acc_both_params={'bias': 0.21716868437244724, 'slope': 19.437646607877017, 'alpha_1': -0.13863158439804657, 'alpha_2': -10.463749682840565, 'gamma': 0.5392507616026618}
## solo optimizanddo gamma
acc_both_gamma = {'gamma': 0.7435064454594396}

##recon_bpsnr lat128_inter_512
params_recon_bpsnr_lat128_inter_512={'gamma': 0.9519615639307367}#optimizacion para recon_bpsnr
params_acc_both_lat128_inter_512 = {'gamma': 0.3012911033494221} # optimización de acc
#acc_both bias y slope y gamma 

results = c.unmix(
    x_test[:n_image],
    x_test_1[:n_image],
    y_test[:n_image],
    y_test_1[:n_image],
    #params={'bias': 0.17786223110878405, 'slope': 24.277471484785064, 'gamma': 0.4665746345952185},
    params = recon_bpsnr_params,
    iterations=10,
    show_image=True,
    show_metrics=False,
    labels=labels
)